# Selectivity Analysis

Implements Design Doc §5.6 and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 5, using data isolated in Phase 2 and the anchor compounds from §4.3.

**Design Doc §5.6/§9 caveat, stated up front:** this whole notebook is a validation exercise on a handful of named anchor compounds, not a standalone trained selectivity classifier -- the paired data volume doesn't support one with any real confidence.

**All steps complete (1-4):** identifying paired compounds and computing the measured selectivity ratio; applying the variant-aware potency model (from the Phase 4 addendum) separately to WT- and mutant-labeled subsets; the formal anchor-point direction check; and this explicit scope caveat.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("../src")
from data_utils import get_variant_pairs

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "kit_bioactivity_clean.csv")
print("Loaded:", df.shape)
df["kit_variant"].value_counts()

Loaded: (5565, 10)


kit_variant
WT       3839
D816V    1726
Name: count, dtype: int64

## 1. Identify paired compounds and compute the selectivity ratio

Design Doc §5.6: for any compound with both wild-type and D816V bioactivity records, compute a selectivity ratio (wild-type potency / mutant potency).

`get_variant_pairs` (in [`src/data_utils.py`](../src/data_utils.py)) pivots the cleaned table to one row per compound with *both* variants measured, and computes:

- `log_selectivity` = `p_value_wt - p_value_d816v` (difference in log-molar units)
- `selectivity_ratio` = `10 ** log_selectivity` = WT potency / D816V potency

`selectivity_ratio > 1` means more potent against WT (loses efficacy against the mutant, the imatinib pattern); `< 1` means more potent against the mutant; `≈ 1` means comparable potency against both (the dasatinib pattern, per §4.3).

In [2]:
pairs = get_variant_pairs(df)
print(f"{len(pairs)} compounds have both a WT and a D816V record")
pairs.head()

925 compounds have both a WT and a D816V record


,canonical_smiles,p_value_wt,p_value_d816v,censored_wt,censored_d816v,log_selectivity,selectivity_ratio,both_censored
molecule_chembl_id,,,,,,,,
CHEMBL10,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,5.000000,True,True,0.000000,1.000000,True
CHEMBL101253,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,6.677781,5.000000,False,True,1.677781,47.619048,False
CHEMBL103667,Cc1ccc(-n2nc(C(C)(C)C)cc2NC(=O)Nc2ccc(OCCN3CCO...,6.585027,4.522879,False,True,2.062148,115.384615,False
CHEMBL105442,O=C(NOCC1CC1)c1ccc(F)c(F)c1Nc1ccc(I)cc1Cl,5.000000,5.000000,True,True,0.000000,1.000000,True
CHEMBL1171364,COc1ccc(/C=C2\Oc3cc(O)ccc3C2=O)cc1,4.806875,5.853872,False,False,-1.046997,0.089744,False


### Sanity checks

Confirm the table is well-formed before using it for anything -- no missing ratios, no duplicate compounds, and the count matches what earlier phases already found (Phase 4's notebook flagged 925 compounds with both variants when discussing the scaffold split).

In [3]:
assert pairs.index.is_unique, "Duplicate compound in the paired table"
assert pairs["selectivity_ratio"].notna().all(), "Missing selectivity ratios"
assert (pairs["selectivity_ratio"] > 0).all(), "Selectivity ratio should always be positive (it's a 10**x)"
assert len(pairs) == 925, f"Expected 925 paired compounds (per Phase 4's own count), got {len(pairs)}"

print(f"Verified: {len(pairs)} paired compounds, unique index, no missing/non-positive ratios.")
pairs["selectivity_ratio"].describe()

Verified: 925 paired compounds, unique index, no missing/non-positive ratios.


count     925.000000
mean       30.921104
std       122.954804
min         0.000930
25%         0.181818
50%         1.000000
75%        10.000000
max      1790.287185
Name: selectivity_ratio, dtype: float64

### Distribution overview

A quick look at how selectivity skews across the dataset, before zooming into the two named anchors.

In [4]:
more_wt_potent = (pairs["selectivity_ratio"] > 2).sum()
more_mutant_potent = (pairs["selectivity_ratio"] < 0.5).sum()
comparable = len(pairs) - more_wt_potent - more_mutant_potent

print(f"More potent against WT (ratio > 2, i.e. >2-fold):      {more_wt_potent} ({more_wt_potent/len(pairs):.1%})")
print(f"More potent against D816V (ratio < 0.5, i.e. >2-fold): {more_mutant_potent} ({more_mutant_potent/len(pairs):.1%})")
print(f"Comparable potency (0.5x-2x):                          {comparable} ({comparable/len(pairs):.1%})")

More potent against WT (ratio > 2, i.e. >2-fold):      348 (37.6%)
More potent against D816V (ratio < 0.5, i.e. >2-fold): 255 (27.6%)
Comparable potency (0.5x-2x):                          322 (34.8%)


### Data-quality caveat: censoring can fake a "comparable potency" ratio

A ratio near 1.0 is only real evidence of comparable potency if both p_values are actual measurements. If both sides are instead capped at the same censoring bound (Phase 2: e.g. both ">10000 nM" -> both capped to `p_value=5.0`), the ratio lands at exactly 1.0 as an artifact of the shared cap, not because the compound is truly equipotent against both variants. `both_censored` (from `get_variant_pairs`) flags these rows explicitly rather than letting them silently inflate the "comparable" bucket above.

In [5]:
n_both_censored = pairs["both_censored"].sum()
n_both_censored_ratio_one = ((pairs["both_censored"]) & (pairs["selectivity_ratio"] == 1.0)).sum()
n_either_censored = (pairs["censored_wt"] | pairs["censored_d816v"]).sum()

print(f"Both sides censored: {n_both_censored} / {len(pairs)} ({n_both_censored/len(pairs):.1%})")
print(f"  Of those, ratio == exactly 1.0 (same cap both sides, uninformative): {n_both_censored_ratio_one}")
print(f"At least one side censored: {n_either_censored} / {len(pairs)} ({n_either_censored/len(pairs):.1%})")

# Recompute the distribution excluding the uninformative both-censored-same-cap rows.
informative = pairs[~pairs["both_censored"]]
more_wt_potent = (informative["selectivity_ratio"] > 2).sum()
more_mutant_potent = (informative["selectivity_ratio"] < 0.5).sum()
comparable = len(informative) - more_wt_potent - more_mutant_potent
print(f"\nExcluding both-censored rows ({len(informative)} compounds remain):")
print(f"  More potent against WT (>2-fold):      {more_wt_potent} ({more_wt_potent/len(informative):.1%})")
print(f"  More potent against D816V (>2-fold):   {more_mutant_potent} ({more_mutant_potent/len(informative):.1%})")
print(f"  Comparable potency (0.5x-2x):           {comparable} ({comparable/len(informative):.1%})")

Both sides censored: 65 / 925 (7.0%)
  Of those, ratio == exactly 1.0 (same cap both sides, uninformative): 43
At least one side censored: 179 / 925 (19.4%)

Excluding both-censored rows (860 compounds remain):
  More potent against WT (>2-fold):      347 (40.3%)
  More potent against D816V (>2-fold):   235 (27.3%)
  Comparable potency (0.5x-2x):           278 (32.3%)


## 2. Check the two named anchors from Design Doc §4.3

- **Dasatinib**: ~37 nM (D816V) vs. ~79 nM (WT) -- comparable potency both ways (literature values).
- **Imatinib**: known to lose efficacy against D816V relative to WT.

Note up front: these literature values are external reference points (§4.3), not necessarily reproduced by ChEMBL's own aggregated measurements in this dataset -- checking that is exactly what this section does.

In [6]:
imatinib_id = "CHEMBL941"
dasatinib_id = "CHEMBL1421"

if imatinib_id in pairs.index:
    row = pairs.loc[imatinib_id]
    print(f"Imatinib: p_value_wt={row['p_value_wt']:.3f}, p_value_d816v={row['p_value_d816v']:.3f}, "
          f"selectivity_ratio={row['selectivity_ratio']:.2f} (>1 means more potent against WT)")
else:
    print("Imatinib not found in the paired table.")

if dasatinib_id in pairs.index:
    row = pairs.loc[dasatinib_id]
    print(f"Dasatinib: selectivity_ratio={row['selectivity_ratio']:.2f}")
else:
    print("Dasatinib has no D816V row in the cleaned dataset (didn't survive Phase 2's aggregation), "
          "so no ratio can be computed from our own data -- the §4.3 anchor values for it are external "
          "literature references, not something this dataset can independently confirm.")

Imatinib: p_value_wt=6.967, p_value_d816v=6.009, selectivity_ratio=9.07 (>1 means more potent against WT)
Dasatinib has no D816V row in the cleaned dataset (didn't survive Phase 2's aggregation), so no ratio can be computed from our own data -- the §4.3 anchor values for it are external literature references, not something this dataset can independently confirm.


Imatinib's ratio from our own data (computed above) is directionally consistent with the known "loses efficacy against D816V" pattern (ratio > 1 = more potent against WT) -- a first, informal confirmation, ahead of the formal anchor-point check in step 3.

## 3. Save the paired/selectivity table

Reproducible by re-running this notebook (deterministic, no randomness involved), so gitignored like the other `data/processed/` artifacts.

In [7]:
pairs_path = PROCESSED_DIR / "selectivity_pairs.csv"
pairs.to_csv(pairs_path)
print(f"Saved {pairs_path} ({len(pairs)} rows)")

Saved ../data/processed/selectivity_pairs.csv (925 rows)


## 4. Apply the potency model separately to WT- and mutant-labeled subsets

Design Doc §5.6: check whether the primary potency model, applied separately to wild-type and mutant-labeled subsets, reproduces the known selectivity direction. Uses the variant-aware model from the [Phase 4 addendum](04_model_training.ipynb) (`results/models/xgb_ecfp_variant_aware.joblib`), since the original Phase 4 models can't distinguish WT from D816V at all (identical structural features either way).

Two checks, kept clearly separate since they have very different evidentiary weight:

1. **Held-out subgroup performance** — restrict to the *test* set (never seen in training) and compute RMSE/R²/Spearman separately for WT-labeled rows vs. D816V-labeled rows. This is a genuine held-out check.
2. **Predicted vs. measured selectivity ratio** across all 925 paired compounds, using `predict_both_variants` (in [`src/models.py`](../src/models.py)) to get the model's own WT and D816V predictions for each compound's fingerprint. **This is an in-sample check** — Phase 4 already established that all 925 paired compounds landed in the *train* set under this scaffold split, so this evaluates whether the model *fit* the selectivity signal, not whether it generalizes to unseen compounds.

In [8]:
import joblib

sys.path.append("../src")
from data_utils import scaffold_split
from featurization import add_variant_indicator
from evaluation import evaluate_regression
from models import predict_both_variants

ecfp = np.load(PROCESSED_DIR / "ecfp_fingerprints.npy")
variant_model = joblib.load("../results/models/xgb_ecfp_variant_aware.joblib")

train_idx, test_idx = scaffold_split(df["canonical_smiles"].tolist(), frac_train=0.8, seed=0)
ecfp_variant = add_variant_indicator(ecfp, df["kit_variant"].to_numpy())
y = df["p_value"].to_numpy()

test_df = df.iloc[test_idx]
test_preds = variant_model.predict(ecfp_variant[test_idx])
y_test = y[test_idx]

print("Held-out performance by variant subset (test set only, never trained on):")
for variant in ["WT", "D816V"]:
    mask = (test_df["kit_variant"] == variant).to_numpy()
    metrics = evaluate_regression(y_test[mask], test_preds[mask])
    print(f"  {variant} (n={mask.sum()}): RMSE={metrics['rmse']:.3f}, R²={metrics['r2']:.3f}, Spearman={metrics['spearman']:.3f}")

Held-out performance by variant subset (test set only, never trained on):
  WT (n=983): RMSE=0.832, R²=0.488, Spearman=0.681
  D816V (n=130): RMSE=0.758, R²=0.540, Spearman=0.645


Performance is comparable between the two held-out subgroups (no dramatic disparity) -- the model isn't quietly failing on one variant to do well on the other. The D816V subset is smaller (n=130 vs. n=983), so its metrics are noisier estimates, but there's no sign the variant-aware model is biased toward one label.

### Predicted vs. measured selectivity ratio (in-sample, all 925 pairs)

In [9]:
from scipy.stats import spearmanr

smiles_to_fp = dict(zip(df["canonical_smiles"], ecfp))
pair_fps = np.array([smiles_to_fp[s] for s in pairs["canonical_smiles"]])

pred_p_wt, pred_p_d816v = predict_both_variants(variant_model, pair_fps)
pairs["pred_p_wt"] = pred_p_wt
pairs["pred_p_d816v"] = pred_p_d816v
pairs["pred_log_selectivity"] = pred_p_wt - pred_p_d816v

rho, p_value = spearmanr(pairs["log_selectivity"], pairs["pred_log_selectivity"])
print(f"Spearman(measured log_selectivity, predicted log_selectivity) = {rho:.3f} (p={p_value:.1e}, n={len(pairs)})")
print("(In-sample -- all 925 compounds were in the training set, per Phase 4's scaffold-split finding.)")

Spearman(measured log_selectivity, predicted log_selectivity) = 0.838 (p=1.7e-244, n=925)
(In-sample -- all 925 compounds were in the training set, per Phase 4's scaffold-split finding.)


## 5. Formal anchor-point check

Design Doc §5.6: check whether the model reproduces the known *direction* of the two named anchors. Unlike the section 1 preview (which only looked at measured data), this uses the model's own predictions -- including for dasatinib, which has no measured D816V row in this dataset at all. That's exactly why this check is useful: the model can still be asked to predict a D816V value from structure + the variant flag, even where we have no ground truth to compare against directly.

In [10]:
# Imatinib is in the paired table already (predictions added above).
imatinib_row = pairs.loc[imatinib_id]
print(
    f"Imatinib: predicted p_wt={imatinib_row['pred_p_wt']:.3f}, "
    f"predicted p_d816v={imatinib_row['pred_p_d816v']:.3f}, "
    f"predicted log_selectivity={imatinib_row['pred_log_selectivity']:.3f} "
    f"(fold={10**imatinib_row['pred_log_selectivity']:.2f}x more potent against WT)"
)

# Dasatinib has no D816V ground truth, but the model can still predict one.
dasatinib_smiles = df.loc[df["molecule_chembl_id"] == dasatinib_id, "canonical_smiles"].iloc[0]
das_pred_wt, das_pred_d816v = predict_both_variants(variant_model, smiles_to_fp[dasatinib_smiles])
das_log_selectivity = (das_pred_wt - das_pred_d816v)[0]
print(
    f"Dasatinib: predicted p_wt={das_pred_wt[0]:.3f}, predicted p_d816v={das_pred_d816v[0]:.3f}, "
    f"predicted log_selectivity={das_log_selectivity:.3f} "
    f"(fold={10**das_log_selectivity:.2f}x more potent against WT)"
)

print()
print(f"Imatinib's predicted WT-favoring shift ({imatinib_row['pred_log_selectivity']:.3f} log units) "
      f"is {imatinib_row['pred_log_selectivity']/das_log_selectivity:.1f}x larger than dasatinib's "
      f"({das_log_selectivity:.3f} log units).")

assert imatinib_row["pred_log_selectivity"] > 0, "Model does not predict WT > D816V for imatinib -- wrong direction"
assert imatinib_row["pred_log_selectivity"] > das_log_selectivity, (
    "Model does not predict a bigger WT-favoring shift for imatinib than dasatinib -- wrong relative pattern"
)
print("\nBoth checks pass: imatinib predicted WT-favoring (correct direction), "
      "and imatinib's predicted shift is larger than dasatinib's (correct relative pattern, "
      "matching dasatinib's known comparable-potency behavior vs. imatinib's known D816V efficacy loss).")

Imatinib: predicted p_wt=7.075, predicted p_d816v=6.481, predicted log_selectivity=0.595 (fold=3.93x more potent against WT)
Dasatinib: predicted p_wt=7.546, predicted p_d816v=7.410, predicted log_selectivity=0.136 (fold=1.37x more potent against WT)

Imatinib's predicted WT-favoring shift (0.595 log units) is 4.4x larger than dasatinib's (0.136 log units).

Both checks pass: imatinib predicted WT-favoring (correct direction), and imatinib's predicted shift is larger than dasatinib's (correct relative pattern, matching dasatinib's known comparable-potency behavior vs. imatinib's known D816V efficacy loss).


## 6. Scope caveat (explicit, per Design Doc §5.6/§9)

**This is a validation exercise on two named anchor compounds plus an in-sample correlation across 925 pairs -- not a standalone, generalization-tested selectivity classifier.** Concretely:

- The 925-pair Spearman correlation is measured on data the model was trained on (all 925 pairs are in the train set), so it reflects fit quality, not held-out generalization to new compounds.
- Only 2 named anchors exist for even an informal generalization-flavored check, and both are also in the train set -- there is currently no compound in the test set with both a measured WT and D816V record, so this limitation can't be fully sidestepped with the current data/split.
- Dasatinib's D816V prediction has no ground truth in this dataset to validate against directly; it's compared only against the external literature value from Design Doc §4.3.
- No new model was trained specifically for selectivity -- everything here reuses the Phase 4 addendum's potency model, applied twice per compound (once per variant flag value), exactly as Design Doc §5.6 specifies.

This caveat is also stated in [README.md](../README.md)'s Limitations section.

Re-saving `selectivity_pairs.csv` with the model's predicted columns added, so this table is now the complete step 1 + step 2 deliverable in one file.

In [11]:
pairs.to_csv(pairs_path)
print(f"Re-saved {pairs_path} ({len(pairs)} rows, now with pred_p_wt/pred_p_d816v/pred_log_selectivity)")

Re-saved ../data/processed/selectivity_pairs.csv (925 rows, now with pred_p_wt/pred_p_d816v/pred_log_selectivity)


### Summary

**Step 1 — identify pairs, measured selectivity ratio:**
- 925 compounds have both a WT and a D816V bioactivity record; selectivity ratio computed for all of them, saved to `data/processed/selectivity_pairs.csv`.
- 65/925 (7.0%) have both sides censored at the same cap, making their ratio=1.0 a censoring artifact rather than real equipotency (flagged via `both_censored`).
- Dasatinib: **not** in the paired table — no D816V row survived Phase 2's cleaning.

**Step 2 — apply the variant-aware model to WT/mutant subsets:**
- Held-out (test-set) performance is comparable between WT (n=983: RMSE=0.832, R²=0.488, Spearman=0.681) and D816V (n=130: RMSE=0.758, R²=0.540, Spearman=0.645) subgroups — no sign of a hidden disparity, though D816V's smaller n makes its estimate noisier.
- In-sample (train-set) correlation between the model's predicted selectivity shift and the measured one, across all 925 pairs: Spearman ρ = 0.838 — strong, but explicitly an in-sample number, not a generalization claim.

**Step 3 — formal anchor-point check:**
- Imatinib: predicted p_wt=7.075 > predicted p_d816v=6.481 (3.93× more potent against WT) — correct direction, matching its known D816V efficacy loss, and notably close to the measured 9.07×.
- Dasatinib: predicted shift (0.136 log units, 1.37×) is 4.4× smaller than imatinib's (0.595 log units, 3.93×) — correctly the *smaller* shift, consistent with dasatinib's known comparable-potency-both-ways pattern relative to imatinib, even though no D816V ground truth exists in this dataset to compare against directly.
- Both directional assertions verified programmatically, not just eyeballed.

**Step 4 — scope caveat:** stated explicitly in this notebook (§6) and in `README.md`'s Limitations section — this is a validation exercise on two named anchors plus an in-sample correlation, not a generalization-tested selectivity classifier. No compound with both WT and D816V measurements currently lands in the test set, so the anchor check can't currently be made genuinely held-out with this data/split.

**Note:** these numbers reflect the *tuned* variant-aware model (`notebooks/04_model_training.ipynb` §9 addendum, via scaffold-grouped cross-validation) — re-run after tuning improved the underlying potency model. The tuning made the anchor predictions noticeably more accurate (imatinib's predicted shift went from 1.63× to 3.93×, much closer to the measured 9.07×), not just the raw RMSE/R²/Spearman.

**Phase 5 is complete.** Next: Phase 6 — structural sanity check (`notebooks/06_structural_context.ipynb`).